# FI-2010 Reproduction — DeepLOB & MLPLOB

**Goal (sprint Day 1):** reproduce the FI-2010 classification baselines and **save results + checkpoints**
(the foundation that was lost in the earlier ephemeral run). Runs DeepLOB and MLPLOB across horizons
k ∈ {10, 20, 50, 100} on the standard CF_7 split, computes full metrics (macro/weighted F1, per-class
P/R, MCC, confusion), runs the acceptance test, and persists everything to S3 / Drive.

### How to use
1. **Runtime → Change runtime type → GPU (T4 is fine).**
2. Run the cells top to bottom. Cell 2 fetches the project code (set `REPO_URL` after you push the repo,
   or use the upload fallback). Cell 4 gets the FI-2010 data (Kaggle or a path you provide).
3. Full run takes roughly **1–3 h on a T4** (8 experiments × early-stopped training).

> Mamba is **not** needed here — DeepLOB and MLPLOB don't import `mamba-ssm`. The Mamba runs come later, in their own notebook.


## 1. Runtime check

In [ ]:
import platform, torch
print("Python:", platform.python_version(), "| Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Runtime > Change runtime type > Hardware accelerator > GPU.")

## 2. Get the project code

The repo is **private**, so the git-clone path needs a token. Options:
- **Colab Secrets** (recommended): add a GitHub PAT (scope `repo`) as a secret named `GH_PAT`
  (key icon in the left sidebar); this cell reads it automatically.
- Or paste a token into `GITHUB_TOKEN` below.
- Or skip cloning and use the **upload fallback** cell (select `models.py`, `fi2010_dataset.py`, `train.py`).

In [ ]:
import sys, subprocess, pathlib

REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"   # private repo
REPO_DIR = "nse-lob-capstone"

GITHUB_TOKEN = ""                       # paste a PAT here, or leave blank to use Colab Secrets
try:
    from google.colab import userdata   # auto-read PAT stored as Colab secret "GH_PAT"
    GITHUB_TOKEN = GITHUB_TOKEN or (userdata.get("GH_PAT") or "")
except Exception:
    pass

def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):          # repo root or modeling/ itself
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None

path = None
for c in (".", REPO_DIR, "/content/" + REPO_DIR):    # already checked out?
    path = add_modeling(c)
    if path:
        break
if not path:                                         # else clone (token for private repo)
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    path = add_modeling(REPO_DIR)

print("modeling/ on sys.path:", path or "NOT FOUND — add a PAT or use the upload fallback cell")

**Upload fallback** (only if you did not git-clone above):

In [ ]:
# from google.colab import files
# files.upload()                       # select models.py, fi2010_dataset.py, train.py
# import sys; sys.path.insert(0, ".")

## 3. Verify dependencies

In [ ]:
import importlib, subprocess, sys
for pkg in ["sklearn", "seaborn", "pandas", "matplotlib", "tqdm", "boto3"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        {"sklearn": "scikit-learn"}.get(pkg, pkg)])
import sklearn, seaborn, pandas, matplotlib  # noqa
print("dependencies ok")

## 4. Get FI-2010 data

The loader expects the **`NoAuction_Zscore`** folder (pre-normalized; do *not* re-normalize), with
`NoAuction_Zscore_Training/` and `NoAuction_Zscore_Testing/` subfolders containing `*CF_7.txt`.

**Recommended (self-contained for the presentation):** the Kaggle cell downloads
`ulfricirons/fi-2010` — the full FI-2010 `BenchmarkDatasets/` archive (~2 GB, includes the
`NoAuction/1.NoAuction_Zscore` folder). It needs a `kaggle.json` API token (Kaggle → Settings →
API → Create New Token). `find_zscore_dir` then locates the right folder automatically.

(Alternatively, if you've put the data on Drive/S3, set `DATA_DIR` and skip the Kaggle cell.)

In [ ]:
import pathlib

DATA_DIR = ""   # <-- set to your NoAuction_Zscore folder if you already have the data

def find_zscore_dir(root):
    """Locate the NoAuction Zscore folder under `root` — the dir holding
    NoAuction_Zscore_Training/ and _Testing/ (what load_fi2010 expects). The
    archive also has an Auction_Zscore folder; we must not pick that one."""
    root = pathlib.Path(root)
    if (root / "NoAuction_Zscore_Training").exists():
        return root
    for p in root.rglob("*"):                        # the actual Training subfolder
        if p.is_dir() and p.name.lower() == "noauction_zscore_training":
            return p.parent
    for p in root.rglob("*"):                         # fallback: a *NoAuction*Zscore* dir
        if p.is_dir() and "zscore" in p.name.lower() and "noauction" in p.name.lower():
            return p
    return None

**Kaggle download** (skip if `DATA_DIR` is already set).

Kaggle auth = a **username** + a **key** (the "API token" string Kaggle gives you). No file needed —
the `kaggle.json` you may have seen is just `{"username": ..., "key": ...}`. Provide them either way:
- **Paste** into `KAGGLE_USERNAME` / `KAGGLE_KEY` below, **or**
- store them as **Colab Secrets** named `KAGGLE_USERNAME` and `KAGGLE_KEY` (auto-read), **or**
- leave both blank to fall back to uploading a `kaggle.json`.

In [ ]:
KAGGLE_SLUG = "ulfricirons/fi-2010"   # FI-2010 benchmark (full BenchmarkDatasets/ structure)
KAGGLE_USERNAME = ""                   # your kaggle handle (kaggle.com/<username>)
KAGGLE_KEY = ""                        # the API token string

if not DATA_DIR:
    import os
    # 1) Colab Secrets (if you stored them there)
    try:
        from google.colab import userdata
        KAGGLE_USERNAME = KAGGLE_USERNAME or (userdata.get("KAGGLE_USERNAME") or "")
        KAGGLE_KEY = KAGGLE_KEY or (userdata.get("KAGGLE_KEY") or "")
    except Exception:
        pass
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
        os.environ["KAGGLE_KEY"] = KAGGLE_KEY
    # 2) else upload a kaggle.json
    elif not os.environ.get("KAGGLE_KEY"):
        from google.colab import files
        print("No KAGGLE_USERNAME/KAGGLE_KEY set — upload kaggle.json instead")
        files.upload()
        !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d {KAGGLE_SLUG} -p fi2010_data --unzip
    # The archive nests a zip that --unzip doesn't recurse into — extract it.
    import zipfile
    for z in list(pathlib.Path("fi2010_data").rglob("*.zip")):
        print("extracting nested zip:", z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(z.parent)
    found = find_zscore_dir("fi2010_data")
    DATA_DIR = str(found) if found else ""

assert DATA_DIR and find_zscore_dir(DATA_DIR), \
    "NoAuction_Zscore folder not found. Set DATA_DIR manually."
DATA_DIR = str(find_zscore_dir(DATA_DIR))
print("DATA_DIR =", DATA_DIR)

## 5. Smoke test — DeepLOB, k=10, 3 epochs (confirm the pipeline learns)

In [ ]:
from fi2010_dataset import load_fi2010
from models import build_model
from train import train, evaluate, DEVICE

tr, vl, te = load_fi2010(DATA_DIR, fold=7, horizon=10, seq_len=100)
m = build_model("deeplob")
print("DeepLOB params:", sum(p.numel() for p in m.parameters()))
_ = train(m, tr, vl, epochs=3, patience=3, batch_size=128, verbose=True)
print("smoke eval (weighted-F1/acc/loss):", evaluate(m, te))

## 6. Metrics helper (full classification report)

In [ ]:
import numpy as np, torch
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             matthews_corrcoef, confusion_matrix)
from fi2010_dataset import make_loader
from train import DEVICE

CLASS_NAMES = ["Down", "Stat", "Up"]

def collect_preds(model, ds, batch_size=256):
    model = model.to(DEVICE).eval()
    loader = make_loader(ds, batch_size=batch_size, shuffle=False)
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(preds), np.concatenate(labels)

def full_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "per_class": {CLASS_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]),
                                       "f1": float(f[i])} for i in range(3)},
        "confusion": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist(),
    }

## 7. Single-experiment runner

Trains one (model, horizon), evaluates with full metrics, saves a checkpoint, and **appends to the
results CSV after every run** (resilient to disconnects). Per-experiment full metrics (per-class +
confusion) are also dumped to JSON.

In [ ]:
import time, json, pathlib, pandas as pd
from fi2010_dataset import load_fi2010
from models import build_model
from train import train, save_checkpoint

RESULTS_DIR = pathlib.Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = pathlib.Path("checkpoints/fi2010"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "fi2010_reproduction.csv"

def run_experiment(model_name, horizon, fold=7, epochs=50, batch_size=128, seq_len=100, lr=1e-3):
    tag = f"{model_name}_cf{fold}_h{horizon}"
    print("=" * 70); print(" ", tag, f"(lr={lr})"); print("=" * 70)
    tr, vl, te = load_fi2010(DATA_DIR, fold=fold, horizon=horizon, seq_len=seq_len)
    model = build_model(model_name)
    n_params = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    hist = train(model, tr, vl, epochs=epochs, batch_size=batch_size, lr=lr, verbose=True)
    elapsed = time.time() - t0
    y_pred, y_true = collect_preds(model, te)
    mt = full_metrics(y_true, y_pred)
    ckpt = CKPT_DIR / f"{tag}.pt"; save_checkpoint(model, str(ckpt))
    row = {"model": model_name, "fold": fold, "horizon": horizon, "n_params": n_params,
           "best_epoch": hist["best_epoch"], "epochs_run": len(hist["val_f1"]),
           "best_val_f1": round(max(hist["val_f1"]), 4),
           "test_accuracy": round(mt["accuracy"], 4),
           "test_macro_f1": round(mt["macro_f1"], 4),
           "test_weighted_f1": round(mt["weighted_f1"], 4),
           "test_mcc": round(mt["mcc"], 4),
           "train_time_s": round(elapsed, 1), "checkpoint": str(ckpt)}
    df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RESULTS_CSV, index=False)
    with open(RESULTS_DIR / f"{tag}_metrics.json", "w") as fjs:
        json.dump(mt, fjs, indent=2)
    print(f"  -> macro_f1={mt['macro_f1']:.4f}  weighted_f1={mt['weighted_f1']:.4f}  "
          f"acc={mt['accuracy']:.4f}  ({elapsed:.0f}s)")
    return row, mt

## 8. Full reproduction run

DeepLOB + MLPLOB + **TLOB** × horizons {10, 20, 50, 100}. `EPOCHS=50` with early stopping (patience 10).
Per-model learning rates follow the papers (TLOB uses 1e-4; the CNN/MLP use 1e-3). It's resumable:
completed `(model, horizon)` rows are skipped, so adding TLOB later and re-running only trains the new
ones (as long as the CSV persists — save it via cell 11, or keep the same runtime).

In [ ]:
MODELS = ["deeplob", "mlplob", "tlob"]
HORIZONS = [10, 20, 50, 100]
EPOCHS = 50
MODEL_LR = {"deeplob": 1e-3, "mlplob": 1e-3, "tlob": 1e-4, "mambalob": 3e-4}

all_metrics = {}
done = set()
if RESULTS_CSV.exists():
    _d = pd.read_csv(RESULTS_CSV)
    done = {(r.model, int(r.horizon)) for r in _d.itertuples()}

for mdl in MODELS:
    for h in HORIZONS:
        if (mdl, h) in done:
            print(f"skip {mdl} h{h} (already in CSV)"); continue
        _, mt = run_experiment(mdl, h, epochs=EPOCHS, lr=MODEL_LR.get(mdl, 1e-3))
        all_metrics[f"{mdl}_h{h}"] = mt

pd.read_csv(RESULTS_CSV)

## 9. Acceptance test

DeepLOB FI-2010 k=10. The paper's headline ~83.4 (Zhang et al. 2019) is the **weighted**-F1 / accuracy-aligned
number (the class set is ~60% Stationary), so we gate acceptance on **weighted-F1 ∈ 78–88%** — matching that
rules out label-mapping / normalization bugs. We **also** report **macro-F1**, the fair imbalanced metric:
~70–75% is expected and matches *independent* reproductions (Prata 2023 / LOBCAST), which found the original
LOB numbers don't fully reproduce. So: paper ↔ our weighted-F1; independent repro ↔ our macro-F1. Report both.

In [ ]:
res = pd.read_csv(RESULTS_CSV)
row = res[(res.model == "deeplob") & (res.horizon == 10)]
if len(row):
    mf1 = float(row.test_macro_f1.iloc[0]) * 100
    wf1 = float(row.test_weighted_f1.iloc[0]) * 100
    print(f"DeepLOB k=10 -> weighted-F1 {wf1:.2f}% (paper ~83.4)  |  macro-F1 {mf1:.2f}%")
    print("ACCEPTANCE (vs paper headline, weighted-F1):",
          "PASS ✓" if 78 <= wf1 <= 88 else "REVIEW ✗ (labels / normalization / epochs)")
    print(f"macro-F1 {mf1:.1f}% — expected ~70-75%, consistent with LOBCAST/Prata 2023 reproductions"
          + (" ✓" if 65 <= mf1 <= 80 else " (outside the usual repro range — inspect)"))
else:
    print("Run DeepLOB h10 first (cell 8).")

## 10. Results table + figures

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, numpy as np

res = pd.read_csv(RESULTS_CSV)
pivot = res.pivot_table(index="horizon", columns="model", values="test_macro_f1")
print(pivot.round(4).to_string())

ax = pivot.plot(marker="o")
ax.set_ylabel("test macro-F1"); ax.set_xlabel("horizon (events)")
ax.set_title("FI-2010 reproduction: macro-F1 by horizon"); ax.grid(True, alpha=0.3)
plt.savefig(RESULTS_DIR / "fig_macrof1_by_horizon.png", dpi=150, bbox_inches="tight"); plt.show()

# Confusion matrices for whatever ran this session
for key, mt in all_metrics.items():
    cm = np.array(mt["confusion"], dtype=float); cmn = cm / cm.sum(1, keepdims=True)
    plt.figure(figsize=(3.2, 2.8))
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(key); plt.ylabel("true"); plt.xlabel("pred"); plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"cm_{key}.png", dpi=150, bbox_inches="tight"); plt.show()

## 11. Persist results + checkpoints

**Do this before the runtime disconnects.** Use Drive (simplest) and/or S3.

In [ ]:
# --- Option A: Google Drive ---
# from google.colab import drive; drive.mount("/content/drive")
# import shutil
# dst = "/content/drive/MyDrive/fi2010_repro"
# shutil.copytree("results", dst + "/results", dirs_exist_ok=True)
# shutil.copytree("checkpoints", dst + "/checkpoints", dirs_exist_ok=True)
# print("copied to", dst)

# --- Option B: S3 (bucket lob-capstone-data, region ap-south-2) ---
import os, glob
S3_BUCKET, S3_PREFIX, S3_REGION = "lob-capstone-data", "reproduction/fi2010", "ap-south-2"
# Provide credentials (or skip this cell and use Drive):
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
try:
    import boto3
    s3 = boto3.client("s3", region_name=S3_REGION)
    for f in glob.glob("results/*") + glob.glob("checkpoints/fi2010/*"):
        key = f"{S3_PREFIX}/{f}"
        s3.upload_file(f, S3_BUCKET, key); print("uploaded", key)
    print(f"Done -> s3://{S3_BUCKET}/{S3_PREFIX}/")
except Exception as e:
    print("S3 upload skipped:", repr(e))

## 12. Reproduction comparison table (ours vs published)

Fill in `PUBLISHED` from the papers before citing. DeepLOB k=10 (~83.4, Zhang 2019) and TLOB
(~92.8 at their settings, Berti & Kasneci 2025) are the anchors; MLPLOB/TLOB other-horizon numbers
come from Berti et al. 2025. Numbers below are **placeholders to verify**, not asserted. Note our
TLOB uses seq_len=100 (the paper's FI-2010 config is seq_size=128) — a documented protocol deviation.

`delta_vs_pub` compares our **weighted**-F1 to the published headline (those numbers are weighted /
accuracy-aligned). We also surface **macro**-F1 — expect it ~10 pts lower and in line with independent
reproductions (LOBCAST / Prata 2023), which is itself a result worth reporting.

In [ ]:
PUBLISHED = {            # avg/weighted F1 (%) as commonly reported — CONFIRM against the papers
    ("deeplob", 10): 83.4,
    ("deeplob", 20): None, ("deeplob", 50): None, ("deeplob", 100): None,
    ("mlplob", 10): None, ("mlplob", 20): None, ("mlplob", 50): None, ("mlplob", 100): None,
    ("tlob", 10): 92.8, ("tlob", 20): None, ("tlob", 50): None, ("tlob", 100): None,
}
res = pd.read_csv(RESULTS_CSV)
res["ours_macro_f1_%"] = (res.test_macro_f1 * 100).round(2)
res["ours_weighted_f1_%"] = (res.test_weighted_f1 * 100).round(2)
res["published_f1"] = res.apply(lambda r: PUBLISHED.get((r.model, int(r.horizon))), axis=1)
res["delta_vs_pub"] = (res["ours_weighted_f1_%"] - res["published_f1"]).round(2)
cols = ["model", "horizon", "ours_macro_f1_%", "ours_weighted_f1_%", "published_f1",
        "delta_vs_pub", "test_accuracy", "best_epoch", "train_time_s"]
res[cols].sort_values(["model", "horizon"])